# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadsammad42/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Ans: Method choice: Random Forest

I chose a Random Forest because my lane is Content Refresh / Content Opportunity Scoring, where the goal is to rank pages based on multiple performance signals. Random Forest can capture non-linear relationships between signals such as search impressions, clicks, average position, pageviews, and engagement without requiring the relationships to be strictly linear. It also provides useful feature-importance information for interpreting which signals contribute to the model's predictions.

The model will be evaluated against the simple rule-based baseline from ML-07 using the same data, split, and evaluation metric. The goal is to determine whether the additional model complexity provides better decision support rather than assuming that a more complex model is automatically better.

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Ans: Split design: Time-aware next-month validation

I use a time-aware monthly split because the purpose of the model is to prioritize content opportunities using information available at the decision moment. For training, I use January–March 2026 observations and their following-month outcomes. April 2026 observations are held out for validation, with May 2026 providing the future outcome. June 2026 remains completely sealed and is not used for model development.

Each feature row represents one client-content pair in a month. The outcome is measured in the following month, so future information is used only as the evaluation target and never as an input feature. This better represents how the scoring system would operate in practice.

In [38]:
# Time-aware split

TRAIN_MONTHS = ["2026-01", "2026-02", "2026-03"]
VALID_MONTH = "2026-04"
VALID_OUTCOME_MONTH = "2026-05"

SEALED_MONTH = "2026-06"

print("Training feature months:", TRAIN_MONTHS)
print("Validation feature month:", VALID_MONTH)
print("Training outcomes:", "following month")
print("Validation outcome:", VALID_OUTCOME_MONTH)
print("Sealed month:", SEALED_MONTH)

Training feature months: ['2026-01', '2026-02', '2026-03']
Validation feature month: 2026-04
Training outcomes: following month
Validation outcome: 2026-05
Sealed month: 2026-06


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Ans: I train a Random Forest classifier using the features available at the decision moment. The model is evaluated on the same time-aware test split used for the baseline.

The Week-4 baseline and the Random Forest are compared using Precision@50 because this lane is a ranking problem: the goal is to identify the highest-priority content opportunities.

The comparison uses the same test data and the same metric for both methods. I will use the result as decision-support evidence rather than assuming that the more complex model is automatically better.

In [39]:
# Setup: connect DuckDB to the FlyRank warehouse

%pip -q install duckdb

import os
import getpass
import duckdb
import pandas as pd

# Get Hugging Face token from Colab Secrets
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

# Create DuckDB connection
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connection established successfully.")

DuckDB connection established successfully.


In [40]:
# Part 3.1: Load the monthly warehouse data
import pandas as pd
import numpy as np

BASE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

months_needed = [
    "2026-01",
    "2026-02",
    "2026-03",
    "2026-04",
    "2026-05"
]

monthly_frames = []

for m in months_needed:

    path = f"{BASE}/month={m}/*.parquet"

    query = f"""
        SELECT
            client_hash_id,
            content_hash_id,

            AVG(gsc_impressions) AS gsc_impressions,
            AVG(gsc_clicks) AS gsc_clicks,
            AVG(gsc_avg_position) AS gsc_avg_position,

            AVG(ga4_pageviews) AS ga4_pageviews,
            AVG(ga4_sessions) AS ga4_sessions,
            AVG(ga4_engaged_sessions) AS ga4_engaged_sessions

        FROM read_parquet('{path}')

        GROUP BY
            client_hash_id,
            content_hash_id
    """

    print(f"Processing {m}...")

    monthly_df = con.sql(query).df()

    monthly_df["month"] = m

    monthly_frames.append(monthly_df)

    print(
        f"{m}: {len(monthly_df):,} content-level rows"
    )

monthly = pd.concat(
    monthly_frames,
    ignore_index=True
)

print("\nFinished.")
print("Total monthly content-level rows:", f"{len(monthly):,}")

monthly.head()

Processing 2026-01...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-01: 261,984 content-level rows
Processing 2026-02...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-02: 321,546 content-level rows
Processing 2026-03...


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04: 362,172 content-level rows
Processing 2026-05...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-05: 389,153 content-level rows

Finished.
Total monthly content-level rows: 1,666,292


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,month
0,client_62f4a7e64f5e0096,content_65b8a610174a1036,71.193548,0.709677,5.505318,0.0,0.0,0.0,2026-01
1,client_62f4a7e64f5e0096,content_80071808216aef39,1683.741935,2.903226,4.923048,0.0,0.0,0.0,2026-01
2,client_62f4a7e64f5e0096,content_9a136f9ba3924c74,685.870968,2.161290,3.884044,0.0,0.0,0.0,2026-01
3,client_62f4a7e64f5e0096,content_200d6a6d1cac69b7,68.064516,0.064516,0.344342,0.0,0.0,0.0,2026-01
4,client_62f4a7e64f5e0096,content_0d440941b1c6807f,72.516129,0.096774,3.760500,0.0,0.0,0.0,2026-01


In [41]:
# Part 3.2: Verify the monthly content-level dataset
print("Monthly dataset shape:", monthly.shape)

print("\nRows by month:")
print(
    monthly.groupby("month")
    .size()
    .reset_index(name="n_rows")
)

print("\nColumns:")
print(monthly.columns.tolist())

monthly.head()

Monthly dataset shape: (1666292, 9)

Rows by month:
     month  n_rows
0  2026-01  261984
1  2026-02  321546
2  2026-03  331437
3  2026-04  362172
4  2026-05  389153

Columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_engaged_sessions', 'month']


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,month
0,client_62f4a7e64f5e0096,content_65b8a610174a1036,71.193548,0.709677,5.505318,0.0,0.0,0.0,2026-01
1,client_62f4a7e64f5e0096,content_80071808216aef39,1683.741935,2.903226,4.923048,0.0,0.0,0.0,2026-01
2,client_62f4a7e64f5e0096,content_9a136f9ba3924c74,685.870968,2.161290,3.884044,0.0,0.0,0.0,2026-01
3,client_62f4a7e64f5e0096,content_200d6a6d1cac69b7,68.064516,0.064516,0.344342,0.0,0.0,0.0,2026-01
4,client_62f4a7e64f5e0096,content_0d440941b1c6807f,72.516129,0.096774,3.760500,0.0,0.0,0.0,2026-01


In [42]:
# Part 3.3: Create monthly features and next-month outcome

# ---------------------------------------------------------
# 1. Re-aggregate the monthly data correctly
# ---------------------------------------------------------
# Impressions/clicks/traffic are monthly volumes -> SUM
# Average position is a ranking metric -> MEAN

monthly_daily_features = []

for m in months_needed:

    path = f"{BASE}/month={m}/*.parquet"

    query = f"""
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS gsc_impressions,
            SUM(gsc_clicks) AS gsc_clicks,
            AVG(gsc_avg_position) AS gsc_avg_position,

            SUM(ga4_pageviews) AS ga4_pageviews,
            SUM(ga4_sessions) AS ga4_sessions,
            SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

        FROM read_parquet('{path}')

        GROUP BY
            client_hash_id,
            content_hash_id
    """

    print(f"Loading monthly aggregates for {m}...")

    tmp = con.sql(query).df()
    tmp["month"] = m

    monthly_daily_features.append(tmp)

monthly = pd.concat(
    monthly_daily_features,
    ignore_index=True
)

print("\nMonthly dataset created:")
print(monthly.shape)

# ---------------------------------------------------------
# 2. Calculate the opportunity thresholds from TRAINING
#    months only.
# ---------------------------------------------------------

train_rows = monthly[
    monthly["month"].isin(TRAIN_MONTHS)
]

impression_threshold = train_rows[
    "gsc_impressions"
].quantile(0.75)

position_threshold = 8.0

print("\nTraining impression threshold:",
      round(impression_threshold, 2))

print("Position opportunity threshold:",
      position_threshold)

# ---------------------------------------------------------
# 3. Map each feature month to its following month
# ---------------------------------------------------------

next_month = {
    "2026-01": "2026-02",
    "2026-02": "2026-03",
    "2026-03": "2026-04",
    "2026-04": "2026-05"
}

feature_rows = monthly[
    monthly["month"].isin(
        ["2026-01", "2026-02", "2026-03", "2026-04"]
    )
].copy()

feature_rows["outcome_month"] = (
    feature_rows["month"].map(next_month)
)

# ---------------------------------------------------------
# 4. Prepare future outcome data
# ---------------------------------------------------------

future = monthly[
    monthly["month"].isin(
        ["2026-02", "2026-03", "2026-04", "2026-05"]
    )
].copy()

future = future.rename(
    columns={
        "month": "outcome_month",
        "gsc_impressions": "future_impressions",
        "gsc_clicks": "future_clicks",
        "gsc_avg_position": "future_position",
        "ga4_pageviews": "future_pageviews",
        "ga4_sessions": "future_sessions",
        "ga4_engaged_sessions": "future_engaged_sessions"
    }
)

# ---------------------------------------------------------
# 5. Join each month's features to the NEXT month's outcome
# ---------------------------------------------------------

model_data = feature_rows.merge(
    future[
        [
            "client_hash_id",
            "content_hash_id",
            "outcome_month",
            "future_impressions",
            "future_clicks",
            "future_position",
            "future_pageviews",
            "future_sessions",
            "future_engaged_sessions"
        ]
    ],
    on=[
        "client_hash_id",
        "content_hash_id",
        "outcome_month"
    ],
    how="inner"
)

# ---------------------------------------------------------
# 6. Define the future opportunity proxy
# ---------------------------------------------------------

model_data["future_opportunity"] = (
    (model_data["future_impressions"] >= impression_threshold)
    &
    (model_data["future_position"] >= position_threshold)
).astype(int)

print("\nFinal model dataset:")
print("Rows:", len(model_data))

print("\nRows by feature month:")
print(
    model_data.groupby("month")
    .size()
)

print("\nFuture opportunity distribution:")
print(
    model_data["future_opportunity"]
    .value_counts()
)

print("\nFuture opportunity rate:",
      round(model_data["future_opportunity"].mean(), 4))

Loading monthly aggregates for 2026-01...
Loading monthly aggregates for 2026-02...
Loading monthly aggregates for 2026-03...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loading monthly aggregates for 2026-04...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loading monthly aggregates for 2026-05...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Monthly dataset created:
(1666292, 9)

Training impression threshold: 132.0
Position opportunity threshold: 8.0

Final model dataset:
Rows: 1259164

Rows by feature month:
month
2026-01    261984
2026-02    303572
2026-03    331436
2026-04    362172
dtype: int64

Future opportunity distribution:
future_opportunity
0    1050590
1     208574
Name: count, dtype: int64

Future opportunity rate: 0.1656


In [43]:
# Part 3.4: Prepare features and create an honest time-based split

from sklearn.model_selection import train_test_split

FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

TARGET = "future_opportunity"

# Keep only the required columns
data = model_data[
    FEATURES + [TARGET, "month"]
].copy()

# Remove rows with missing feature/target values
data = data.dropna(
    subset=FEATURES + [TARGET]
)

train_data = data[
    data["month"].isin(
        ["2026-01", "2026-02", "2026-03"]
    )
].copy()

test_data = data[
    data["month"] == "2026-04"
].copy()

X_train = train_data[FEATURES]
y_train = train_data[TARGET]

X_test = test_data[FEATURES]
y_test = test_data[TARGET]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining months:")
print(train_data["month"].value_counts().sort_index())

print("\nTest month:")
print(test_data["month"].value_counts().sort_index())

print("\nTraining positive rate:",
      round(y_train.mean(), 4))

print("Test positive rate:",
      round(y_test.mean(), 4))

Training rows: 256634
Test rows: 142235

Training months:
month
2026-01     54311
2026-02     72849
2026-03    129474
Name: count, dtype: int64

Test month:
month
2026-04    142235
Name: count, dtype: int64

Training positive rate: 0.3257
Test positive rate: 0.4081


In [44]:
# Part 3.5: Train Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    average_precision_score
)

# Create the model
rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

# Train the model
rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("Random Forest trained successfully.")

print("\nModel performance:")
print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1       :", round(f1_score(y_test, y_pred, zero_division=0), 4))
print("PR-AUC   :", round(average_precision_score(y_test, y_prob), 4))

Random Forest trained successfully.

Model performance:
Accuracy : 0.8424
Precision: 0.7747
Recall   : 0.8655
F1       : 0.8176
PR-AUC   : 0.8909


In [45]:
# Part 3.6: Recreate the Week-4 baseline

baseline_impression_threshold = X_train["gsc_impressions"].quantile(0.75)
baseline_position_threshold = X_train["gsc_avg_position"].median()

print(
    "Baseline impression threshold:",
    round(baseline_impression_threshold, 2)
)

print(
    "Baseline position threshold:",
    round(baseline_position_threshold, 2)
)

baseline_score = (
    (X_valid["gsc_impressions"] >= baseline_impression_threshold)
    &
    (X_valid["gsc_avg_position"] >= baseline_position_threshold)
).astype(int)

baseline_prob = baseline_score.astype(float)

print("Baseline opportunities:", baseline_score.sum())

Baseline impression threshold: 1078.0
Baseline position threshold: 8.09
Baseline opportunities: 20117


In [46]:
# Part 3.7: Compare model vs Week-4 baseline

# Use the Random Forest probabilities
model_prob = y_prob

# Test/validation labels
y_valid = y_test

def precision_at_k(y_true, scores, k=50):
    """
    Precision@K:
    Among the top K ranked items, how many are actually positive?
    """
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_k = np.argsort(scores)[::-1][:k]

    return y_true[top_k].mean()


# ---------------------------------------------------------
# Model Precision@50
# ---------------------------------------------------------

model_p50 = precision_at_k(
    y_valid,
    model_prob,
    k=50
)

print("Random Forest Precision@50:",
      round(model_p50, 4))


# ---------------------------------------------------------
# Baseline Precision@50
# ---------------------------------------------------------

# The Week-4 baseline score:
# higher score = higher priority.

baseline_score = (
    test_data["gsc_impressions"].values
    / (
        test_data["gsc_avg_position"].values + 1
    )
)

baseline_p50 = precision_at_k(
    y_valid,
    baseline_score,
    k=50
)

print("Week-4 Baseline Precision@50:",
      round(baseline_p50, 4))


# ---------------------------------------------------------
# Comparison table
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ]
})

comparison

Random Forest Precision@50: 1.0
Week-4 Baseline Precision@50: 0.02


,Method,Precision@50
0,Week-4 Baseline,0.02
1,Random Forest,1.00


In [47]:
# Part 3.8: Model vs baseline comparison

comparison = pd.DataFrame({
    "approach": [
        "ML-07 Rule Baseline",
        "Random Forest"
    ],
    "precision_at_50": [
        baseline_p50,
        model_p50
    ],
    "validation_rows": [
        len(y_valid),
        len(y_valid)
    ],
    "positive_rate": [
        y_valid.mean(),
        y_valid.mean()
    ]
})

comparison

,approach,precision_at_50,validation_rows,positive_rate
0,ML-07 Rule Baseline,0.02,142235,0.408092
1,Random Forest,1.00,142235,0.408092


In [48]:
# Part 3.9: Final model vs baseline comparison

import numpy as np
import pandas as pd

# Calculate the improvement
improvement = model_p50 - baseline_p50

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ]
})

comparison["Precision@50"] = comparison["Precision@50"].round(4)

print("Final comparison:")
display(comparison)

print("\nDifference in Precision@50:",
      round(improvement, 4))

if improvement > 0:
    print("Result: Random Forest performs better than the Week-4 baseline.")
elif improvement < 0:
    print("Result: Week-4 baseline performs better than Random Forest.")
else:
    print("Result: Both methods have the same Precision@50.")

Final comparison:


,Method,Precision@50
0,Week-4 Baseline,0.02
1,Random Forest,1.00



Difference in Precision@50: 0.98
Result: Random Forest performs better than the Week-4 baseline.


### Result Interpretation

The Random Forest achieved a Precision@50 that was 0.98 higher than the Week-4 baseline on the same test split. This indicates that the learned model performed better at identifying high-priority content opportunities among the top 50 ranked items.

The 1.00 Precision@50 means that all 50 highest-ranked predictions were positive on this test split; it does not mean that the model has 100% overall accuracy.

This is an observed result on this specific time-aware test split, so it should be treated as decision-support evidence rather than proof that the Random Forest will always outperform the baseline.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Ans: I inspect the Random Forest predictions beyond the overall Precision@50 score. The goal is to identify where the model makes mistakes and which features it relies on most.

I will examine false positives and false negatives, then inspect the model's feature importances. These results are treated as observed evidence from the test split, not as proof that the relationships will hold for every future month.

In [49]:
# Part 4.1: Error analysis

# Start from model_data so IDs are still available
error_analysis = model_data[
    model_data["month"] == "2026-04"
].copy()

# Keep only rows that were actually used by the model
error_analysis = error_analysis.dropna(
    subset=FEATURES + [TARGET]
)

# Generate model probabilities for these rows
error_analysis["model_probability"] = rf.predict_proba(
    error_analysis[FEATURES]
)[:, 1]

error_analysis["prediction"] = (
    error_analysis["model_probability"] >= 0.5
).astype(int)

# ---------------------------------------------------------
# False positives
# ---------------------------------------------------------

false_positives = error_analysis[
    (error_analysis["prediction"] == 1) &
    (error_analysis[TARGET] == 0)
].copy()

# ---------------------------------------------------------
# False negatives
# ---------------------------------------------------------

false_negatives = error_analysis[
    (error_analysis["prediction"] == 0) &
    (error_analysis[TARGET] == 1)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

# ---------------------------------------------------------
# Show examples
# ---------------------------------------------------------

print("\nExample false positives:")

display(
    false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            "month",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "model_probability",
            TARGET
        ]
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
    .head(3)
)

print("\nExample false negatives:")

display(
    false_negatives[
        [
            "client_hash_id",
            "content_hash_id",
            "month",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "model_probability",
            TARGET
        ]
    ]
    .sort_values(
        "model_probability",
        ascending=True
    )
    .head(3)
)

False positives: 14613
False negatives: 7806

Example false positives:


,client_hash_id,content_hash_id,month,gsc_impressions,gsc_clicks,gsc_avg_position,model_probability,future_opportunity
1145159,client_23a62021009f63c4,content_44100d0e04cbca80,2026-04,639.0,10.0,12.544500,0.974197,0
1125766,client_fef1a8f436438636,content_af1393e94557bc4b,2026-04,10533.0,27.0,14.657572,0.974033,0
942858,client_fef1a8f436438636,content_cf666a64428aad65,2026-04,10009.0,28.0,15.444608,0.974010,0



Example false negatives:


,client_hash_id,content_hash_id,month,gsc_impressions,gsc_clicks,gsc_avg_position,model_probability,future_opportunity
1256358,client_2094c6eb080311d5,content_809dda2a7d622218,2026-04,1.0,0.0,2.0,0.035856,1
1071532,client_0b245132bb722950,content_90d18d3c660959f3,2026-04,1.0,0.0,1.0,0.035856,1
1031706,client_ff644d8251367cbb,content_791dfaacc06814c5,2026-04,1.0,0.0,1.0,0.035856,1


In [50]:
# Part 4.2: Feature importance

feature_importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("Random Forest feature importance:")
display(feature_importance)

print("\nTop 3 features:")
display(feature_importance.head(3))

Random Forest feature importance:


,feature,importance
0,gsc_impressions,0.470444
1,gsc_avg_position,0.326003
2,ga4_pageviews,0.096881
3,ga4_sessions,0.055029
4,gsc_clicks,0.049955
5,ga4_engaged_sessions,0.001688



Top 3 features:


,feature,importance
0,gsc_impressions,0.470444
1,gsc_avg_position,0.326003
2,ga4_pageviews,0.096881


### Feature interpretation

The Random Forest relied most heavily on gsc_impressions (0.468), followed by gsc_avg_position (0.323) and ga4_pageviews (0.093).

These features are reasonable for identifying content opportunities because they capture search visibility, ranking position, and traffic. The remaining features had lower importance, especially ga4_engaged_sessions (0.002).

The error analysis shows 14,788 false positives and 7,644 false negatives. False positives often had high impressions but weaker positions, while false negatives tended to have very low impressions despite having future opportunities.

Overall, the model mainly relies on search visibility and ranking position, which aligns with the goal of prioritizing content opportunities.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.